# 03. Google Earth Engine (GEE) Covariate Extraction & Remote Sensing Fusion

**Objective**: Extract multi-sensor satellite covariates for all **5,689 spring points** clipped to the official **OpenStreetMap Kavrepalanchok District Polygon**:
1. **SRTM 30m DEM**: Elevation ($m$), Slope ($^\circ$), Aspect ($^\circ$), Topographic Wetness Index (TWI)
2. **Sentinel-2 (10m)**: Cloud-masked multi-year median composite: NDVI mean, NDVI seasonal std, NDWI moisture
3. **CHIRPS Daily (5.5km)**: Annual mean precipitation ($mm$), Monsoon precipitation ($mm$), Dry-season precipitation ($mm$)

### Pipeline Order
- `01_tabular_eda.ipynb` (Statistical & Survey EDA)
- `02_geospatial_eda.ipynb` (Spatial Patterns, OSM Boundaries & Maps)
- `03_gee_extraction.ipynb` (Remote Sensing & Satellite Fusion)

## Step 1: Initialize Earth Engine & Load Official District Boundary

In [1]:
import ee
import geemap
import geopandas as gpd
import pandas as pd
import json
from pathlib import Path

# Initialize Earth Engine (authenticated)
ee.Initialize()
print('Google Earth Engine successfully initialized!')

# Load official OpenStreetMap Kavre district polygon
with open('../data/processed/kavre_district_boundary.geojson', 'r', encoding='utf-8') as f:
    boundary_json = json.load(f)

kavre_geom = ee.Geometry(boundary_json['features'][0]['geometry'])
print('Loaded official Kavre boundary geometry into GEE.')

Google Earth Engine successfully initialized!
Loaded official Kavre boundary geometry into GEE.


## Step 2: Interactive Terrain & Satellite Visualization (geemap)

In [2]:
Map = geemap.Map(center=[27.57, 85.61], zoom=11)

# Clip SRTM 30m directly to official Kavre polygon
srtm = ee.Image('USGS/SRTMGL1_003').clip(kavre_geom)
hillshade = ee.Terrain.hillshade(srtm)

Map.addLayer(hillshade, {}, 'Hillshade')
Map.addLayer(srtm, {'min': 600, 'max': 3000, 'palette': ['#2b83ba', '#abdda4', '#ffffbf', '#fdae61', '#d7191c']}, 'SRTM Elevation (m)')
Map.addLayer(kavre_geom, {'color': 'black'}, 'Kavre Official Border')
Map

Map(center=[27.57, 85.61], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright'…

## Step 3: Run Batch Covariate Extraction
Samples all multi-band satellite rasters for all 5,689 coordinates.

In [3]:
import sys
sys.path.append('..')
from src.gee_extraction import extract_gee_covariates

extracted_df = extract_gee_covariates(
    input_csv=Path('../data/processed/clean_springs.csv'),
    output_csv=Path('../data/processed/gee_features.csv'),
    batch_size=1000
)
extracted_df.head()

2026-11-05 10:53:35,312 [INFO] Initializing Google Earth Engine...
2026-11-05 10:53:37,596 [INFO] GEE successfully initialized!
2026-11-05 10:53:37,634 [INFO] Loaded 5689 spring points from ..\data\processed\clean_springs.csv
2026-11-05 10:53:37,636 [INFO] Using bounding box geometry for GEE clipping.
2026-11-05 10:53:37,637 [INFO] Constructing raster covariates composite...
2026-11-05 10:53:38,334 [INFO] Extracting 10 bands: ['elevation_m', 'slope_deg', 'aspect_deg', 'twi', 'ndvi_mean', 'ndvi_std', 'ndwi_mean', 'precip_annual_mm', 'precip_monsoon_mm', 'precip_dry_season_mm']
2026-11-05 10:53:38,336 [INFO] Processing batch 0 to 1000...
2026-11-05 10:53:41,138 [INFO] Processing batch 1000 to 2000...
2026-11-05 10:53:44,365 [INFO] Processing batch 2000 to 3000...
2026-11-05 10:53:48,924 [INFO] Processing batch 3000 to 4000...
2026-11-05 10:57:15,475 [INFO] Processing batch 4000 to 5000...
2026-11-05 10:59:11,652 [INFO] Processing batch 5000 to 5689...
2026-11-05 10:59:36,845 [INFO] GEE e

,aspect_deg,elevation_m,id,ndvi_mean,ndvi_std,ndwi_mean,precip_annual_mm,precip_dry_season_mm,precip_monsoon_mm,slope_deg,twi
0,257.503082,1655,1,0.668009,0.145982,-0.625184,1358.817306,40.362925,1077.587198,27.639465,1.340051
1,90.000000,1197,2,0.339442,0.113182,-0.443882,1291.633665,40.147142,1016.021649,16.286755,1.923568
2,226.975250,1473,3,0.352935,0.104595,-0.427816,1426.238758,41.907239,1132.611324,25.384155,2.131796
3,319.772552,1481,4,0.738402,0.140402,-0.655337,1426.238758,41.907239,1132.611324,4.847637,3.853640
4,296.869202,1479,5,0.675196,0.147597,-0.605421,1426.238758,41.907239,1132.611324,8.153214,3.329327


## Step 4: Merge Ground-Truth Survey + Satellite Features
Produces `master_dataset.parquet` for machine learning and Graph Neural Network modeling.

In [4]:
survey_df = pd.read_csv('../data/processed/clean_springs.csv')
gee_df = pd.read_csv('../data/processed/gee_features.csv')

master_df = survey_df.merge(gee_df, on='id', how='left')
master_df.to_parquet('../data/processed/master_dataset.parquet', index=False)
print(f"Master dataset assembled: {master_df.shape[0]} rows, {master_df.shape[1]} features")
master_df.head(3)

Master dataset assembled: 5689 rows, 95 features


,id,municipality,ward,village,latitude,longitude,source_type,ownership,landuse,source_condition,...,aspect_deg,elevation_m,ndvi_mean,ndvi_std,ndwi_mean,precip_annual_mm,precip_dry_season_mm,precip_monsoon_mm,slope_deg,twi
0,1,Namobuudha,11,Thalbale,27.556629,85.596365,spring,community,agriculture,active,...,257.503082,1655,0.668009,0.145982,-0.625184,1358.817306,40.362925,1077.587198,27.639465,1.340051
1,2,Namobuudha,11,Bimire,27.565024,85.638463,spring,community,agriculture,dried,...,90.000000,1197,0.339442,0.113182,-0.443882,1291.633665,40.147142,1016.021649,16.286755,1.923568
2,3,Namobuudha,11,Nagar,27.544086,85.569624,spring,community,agriculture,active,...,226.975250,1473,0.352935,0.104595,-0.427816,1426.238758,41.907239,1132.611324,25.384155,2.131796
